In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import numpy as np
import cv2
from PIL import Image
import re
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import time
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os
import warnings
import random
from typing import Optional
warnings.filterwarnings('ignore')


In [24]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class VideoFramesDataset(Dataset):
    def __init__(self, 
                 real_root: str, 
                 fake_root: str,
                 frames_per_video: int = 32, 
                 frame_size: int = 224,
                 is_train: bool = False, 
                 seed: int = 42,
                 max_real: Optional[int] = None,
                 max_fake: Optional[int] = None):
        
        self.frames_per_video = frames_per_video
        self.frame_size = frame_size
        self.is_train = is_train
        self.rng = random.Random(seed) if not is_train else random
        
        real_root, fake_root = Path(real_root), Path(fake_root)
        real_paths = sorted([d for d in real_root.iterdir() if d.is_dir()]) if real_root.exists() else []
        fake_paths = sorted([d for d in fake_root.iterdir() if d.is_dir()]) if fake_root.exists() else []
        
        print(f"Scanning {real_root.name}: found {len(real_paths)} real videos")
        print(f"Scanning {fake_root.name}: found {len(fake_paths)} fake videos")

        if max_real is not None and len(real_paths) > max_real:
            self.rng.shuffle(real_paths)
            real_paths = real_paths[:max_real]
        if max_fake is not None and len(fake_paths) > max_fake:
            self.rng.shuffle(fake_paths)
            fake_paths = fake_paths[:max_fake]
            
        print(f"После фильтрации: {len(real_paths)} real, {len(fake_paths)} fake")

        self.video_paths = real_paths + fake_paths
        self.labels = [0] * len(real_paths) + [1] * len(fake_paths)
        
        print(f"Total {len(self.video_paths)} videos for {'train' if is_train else 'val/test'} split")
        print(f"Real (0): {self.labels.count(0)}")
        print(f"Fake (1): {self.labels.count(1)}")

        if not is_train:
            combined = list(zip(self.video_paths, self.labels))
            self.rng.shuffle(combined)
            self.video_paths, self.labels = zip(*combined)
            self.video_paths = list(self.video_paths)
            self.labels = list(self.labels)

        if self.is_train:
            self.transform = transforms.Compose([
                transforms.RandomResizedCrop(self.frame_size, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5),
                transforms.RandomGrayscale(p=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(self.frame_size + 32),
                transforms.CenterCrop(self.frame_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
            ])

    def __len__(self):
        return len(self.video_paths)
    
    def load_frames(self, video_path: Path):
        frames = []
        frame_files = sorted([f for f in video_path.glob('*.jpg')])
        
        if len(frame_files) > self.frames_per_video:
            indices = np.linspace(0, len(frame_files)-1, self.frames_per_video, dtype=int)
            frame_files = [frame_files[i] for i in indices]
         
        for frame_file in frame_files:
            try:
                img = Image.open(frame_file).convert('RGB')
            except Exception:
                img = Image.new('RGB', (self.frame_size, self.frame_size))
            frames.append(img)

        if len(frames) > 0:
            orig_len = len(frames)
            while len(frames) < self.frames_per_video:
                frames.append(frames[len(frames) % orig_len])
        else:
            while len(frames) < self.frames_per_video:
                frames.append(Image.new('RGB', (self.frame_size, self.frame_size)))
        
        return frames
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_frames(video_path)
        
        transformed_frames = [self.transform(frame) for frame in frames]
        frames_tensor = torch.stack(transformed_frames)
        
        return frames_tensor, torch.tensor(label, dtype=torch.long)


def create_dataloaders(root_dir: str,
                       batch_size: int = 4, 
                       num_workers: int = 2, 
                       frames_per_video: int = 16,
                       val_max_real=None, val_max_fake=None,
                       test_max_real=None, test_max_fake=None,
                       seed: int = 42):
    """
    Создаёт Даталоадеры. Ожидается структура:
    root_dir/
      train_real/ (содержит папки с кадрами видео)
      train_fake/
      val_real/
      val_fake/
      test_real/
      test_fake/
    """
    root = Path(root_dir)

    train_dataset = VideoFramesDataset(real_root=root / 'train_real', fake_root=root / 'train_fake', frames_per_video=frames_per_video, is_train=True, seed=seed)
    
    val_dataset = VideoFramesDataset(real_root=root / 'val_real', fake_root=root / 'val_fake', frames_per_video=frames_per_video, is_train=False, seed=seed, 
                                     max_real=val_max_real, max_fake=val_max_fake)
    
    test_dataset = VideoFramesDataset(real_root=root / 'test_real', fake_root=root / 'test_fake', frames_per_video=frames_per_video, is_train=False, seed=seed, 
                                      max_real=test_max_real, max_fake=test_max_fake)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0)
    
    return train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset

In [41]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
    
ROOT_DIR = "/kaggle/input/datasets/mariaspasyuk"  

train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = create_dataloaders(root_dir='/kaggle/input/datasets/mariaspasyuk/eye-features',
                       batch_size = 32, 
                       num_workers = 2, 
                       frames_per_video = 16,
                       val_max_real=None, val_max_fake=200,
                       test_max_real=None, test_max_fake=500,
                       seed = 42)

Using device: cuda
Scanning train_real: found 2860 real videos
Scanning train_fake: found 3444 fake videos
 После фильтрации: 2860 real, 3444 fake
 Total 6304 videos for train split
   - Real (0): 2860
   - Fake (1): 3444
Scanning val_real: found 147 real videos
Scanning val_fake: found 286 fake videos
 После фильтрации: 147 real, 200 fake
 Total 347 videos for val/test split
   - Real (0): 147
   - Fake (1): 200
Scanning test_real: found 421 real videos
Scanning test_fake: found 992 fake videos
 После фильтрации: 421 real, 500 fake
 Total 921 videos for val/test split
   - Real (0): 421
   - Fake (1): 500


In [32]:
class ResNetLSTM(nn.Module):
    def __init__(self, lstm_hidden=256, lstm_layers=2, dropout=0.2, 
                 freeze_cnn=True, unfreeze_layers=0):
        super().__init__()

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2 
        self.layer3 = resnet.layer3  
        self.layer4 = resnet.layer4  
        
        self.avgpool = resnet.avgpool

        self.cnn_layers = [self.layer1, self.layer2, self.layer3, self.layer4]
        
        self.cnn_out_dim = 2048

        if freeze_cnn:
            for param in self.conv1.parameters():
                param.requires_grad = False
            for param in self.bn1.parameters():
                param.requires_grad = False
            for layer in self.cnn_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        if unfreeze_layers > 0:
            layers_to_unfreeze = self.cnn_layers[-unfreeze_layers:]
            for i, layer in enumerate(layers_to_unfreeze):
                for param in layer.parameters():
                    param.requires_grad = True
            print(f"Разморожено слоев: {unfreeze_layers}")
            print(f"({[f'layer{4-j}' for j in range(len(layers_to_unfreeze))][::-1]})")
        
        self.lstm = nn.LSTM(
            input_size=self.cnn_out_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        batch_size, frames, C, H, W = x.shape
        
        cnn_out = []
        for t in range(frames):
            frame = x[:, t, :, :, :]
            
            feat = self.conv1(frame)
            feat = self.bn1(feat)
            feat = self.relu(feat)
            feat = self.maxpool(feat)
            
            feat = self.layer1(feat)
            feat = self.layer2(feat)
            feat = self.layer3(feat)
            feat = self.layer4(feat)
            
            feat = self.avgpool(feat)
            feat = feat.view(batch_size, -1)
            cnn_out.append(feat)
            
        cnn_seq = torch.stack(cnn_out, dim=1)

        lstm_out, _ = self.lstm(cnn_seq)
        last_out = lstm_out[:, -1, :]
        logits = self.classifier(last_out)
        
        return logits

In [33]:
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def train_epoch_ft(model, loader, criterion, optimizer, device, max_grad_norm=1.0):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for frames, labels in tqdm(loader, desc='Train'):
        frames, labels = frames.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(frames)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        
        optimizer.step()
        
        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    acc = accuracy_score(all_labels, all_preds)
    return running_loss / len(loader), acc

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for frames, labels in tqdm(loader, desc='Val'):
            frames, labels = frames.to(device), labels.to(device)
            logits = model(frames)
            loss = criterion(logits, labels)
            running_loss += loss.item()
            
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    return running_loss / len(loader), acc, f1, auc

In [7]:
import os
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

print("Fine-tuning")

model = ResNetLSTM(lstm_hidden=256,lstm_layers=2, dropout=0.6, freeze_cnn=True, unfreeze_layers=1).to(device)

print("layer1 requires_grad:", next(model.layer1.parameters()).requires_grad)
print("layer2 requires_grad:", next(model.layer2.parameters()).requires_grad)
print("layer3 requires_grad:", next(model.layer3.parameters()).requires_grad)
print("layer4 requires_grad:", next(model.layer4.parameters()).requires_grad)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-6, weight_decay=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
CHECKPOINT_DIR = Path("checkpoints_finetuned")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
tb_writer = SummaryWriter(log_dir='runs/fine-tuned_cnn')

best_val_auc = 0.0
train_loss_1, train_acc_1 = [], []
val_loss_1, val_acc_1, val_f1_1, val_auc_1 = [], [], [], []
EPOCHS = 10

for epoch in range(EPOCHS):
    torch.cuda.empty_cache()
    train_loss, train_acc = train_epoch_ft(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, val_auc = validate_epoch(model, val_loader, criterion, device)
    
    train_loss_1.append(round(train_loss, 4)); train_acc_1.append(round(train_acc, 4))
    val_loss_1.append(round(val_loss, 4)); val_acc_1.append(round(val_acc, 4))
    val_f1_1.append(round(val_f1, 4)); val_auc_1.append(round(val_auc, 4))
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    tb_writer.add_scalar('Stage2/Loss/Train', train_loss, epoch+1)
    tb_writer.add_scalar('Stage2/Accuracy/Train', train_acc, epoch+1)
    tb_writer.add_scalar('Stage2/Learning_Rate', current_lr, epoch+1)
    tb_writer.add_scalar('Stage2/Loss/Val', val_loss, epoch+1)
    tb_writer.add_scalar('Stage2/Accuracy/Val', val_acc, epoch+1)
    tb_writer.add_scalar('Stage2/F1/Val', val_f1, epoch+1)
    tb_writer.add_scalar('Stage2/AUC/Val', val_auc, epoch+1)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Acc={train_acc:.4f}, "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model_fine-tuned.pth")
        
    torch.save(model.state_dict(), CHECKPOINT_DIR / f"model_fine-tuned_epoch_{epoch+1}.pth")

tb_writer.close()
print("Логи сохранены в папку `runs/fine-tuned_cnn`")

2026-05-24 07:52:40.141637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779609160.359394      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779609160.426563      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779609160.946248      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779609160.946291      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779609160.946294      57 computation_placer.cc:177] computation placer alr


=== Fine-tuning (selective unfreezing) ===
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s] 


Разморожено слоев: 1
   (['layer4'])
layer1 requires_grad: False
layer2 requires_grad: False
layer3 requires_grad: False
layer4 requires_grad: True


Val: 100%|██████████| 22/22 [00:35<00:00,  1.61s/it]


...

Epoch 8: Train Loss=0.5848 Acc=0.7874 | Val Loss=0.5718 Acc=0.7723 F1=0.7836 AUC=0.8603


Val: 100%|██████████| 22/22 [00:19<00:00,  1.13it/s]


Epoch 9: Train Loss=0.5599 Acc=0.8074 | Val Loss=0.5693 Acc=0.7550 F1=0.8132 AUC=0.8665


Val: 100%|██████████| 22/22 [00:21<00:00,  1.04it/s]


Epoch 10: Train Loss=0.5410 Acc=0.8209 | Val Loss=0.5359 Acc=0.8098 F1=0.8316 AUC=0.8746
Логи сохранены в папку `runs/fine-tuned_cnn`


In [8]:
test_loss, test_acc, test_f1, test_auc = validate_epoch(model, test_loader, criterion, device)
print(f"Test Results: Loss={test_loss:.4f} Acc={test_acc:.4f} F1={test_f1:.4f} AUC={test_auc:.4f}")

Val: 100%|██████████| 58/58 [01:29<00:00,  1.54s/it]

Test Results: Loss=0.5120 Acc=0.8350 F1=0.8449 AUC=0.9071


In [19]:
for epoch in range(10, 15):
    torch.cuda.empty_cache()
    train_loss, train_acc = train_epoch_ft(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, val_auc = validate_epoch(model, val_loader, criterion, device)
    
    train_loss_1.append(round(train_loss, 4)); train_acc_1.append(round(train_acc, 4))
    val_loss_1.append(round(val_loss, 4)); val_acc_1.append(round(val_acc, 4))
    val_f1_1.append(round(val_f1, 4)); val_auc_1.append(round(val_auc, 4))
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    tb_writer.add_scalar('Stage2/Loss/Train', train_loss, epoch+1)
    tb_writer.add_scalar('Stage2/Accuracy/Train', train_acc, epoch+1)
    tb_writer.add_scalar('Stage2/Learning_Rate', current_lr, epoch+1)
    tb_writer.add_scalar('Stage2/Loss/Val', val_loss, epoch+1)
    tb_writer.add_scalar('Stage2/Accuracy/Val', val_acc, epoch+1)
    tb_writer.add_scalar('Stage2/F1/Val', val_f1, epoch+1)
    tb_writer.add_scalar('Stage2/AUC/Val', val_auc, epoch+1)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Acc={train_acc:.4f}, "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model_fine-tuned.pth")
        
    torch.save(model.state_dict(), CHECKPOINT_DIR / f"model_fine-tuned_epoch_{epoch+1}.pth")

tb_writer.close()
print("Логи сохранены в папку `runs/fine-tuned_cnn`")

Val: 100%|██████████| 22/22 [00:19<00:00,  1.10it/s]


Epoch 11: Train Loss=0.4979 Acc=0.8453, Val Loss=0.5101 Acc=0.8184 F1=0.8346 AUC=0.8832


Val: 100%|██████████| 22/22 [00:20<00:00,  1.10it/s]


Epoch 12: Train Loss=0.4812 Acc=0.8598, Val Loss=0.5105 Acc=0.8156 F1=0.8476 AUC=0.8896


Val: 100%|██████████| 22/22 [00:20<00:00,  1.06it/s]/it]


Epoch 13: Train Loss=0.4734 Acc=0.8580, Val Loss=0.4901 Acc=0.8156 F1=0.8325 AUC=0.8925


Val: 100%|██████████| 22/22 [00:21<00:00,  1.02it/s]


Epoch 14: Train Loss=0.4610 Acc=0.8648, Val Loss=0.4904 Acc=0.8127 F1=0.8338 AUC=0.8900


Val: 100%|██████████| 22/22 [00:23<00:00,  1.09s/it]

Epoch 15: Train Loss=0.4523 Acc=0.8658, Val Loss=0.4921 Acc=0.7954 F1=0.8212 AUC=0.8902
Логи сохранены в папку `runs/fine-tuned_cnn`


In [5]:
import os
import pandas as pd 
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image 
import torchvision.transforms as T
import numpy as np
from pathlib import Path

class HuaweiTestDataset(Dataset):
    def __init__(self, csv_path, frames_root, num_frames=16, transform=None):
        self.df = pd.read_csv(csv_path)
        self.frames_root = Path(frames_root)
        self.num_frames = num_frames
        self.transform = transform
        
        self.video_data = []
        
        for _, row in self.df.iterrows():
            obj_id = str(row['obj_id'])
            label = int(row['label']) if not pd.isna(row['label']) else -1
            
            video_dir = self.frames_root / obj_id
            if video_dir.exists():
                frames = list(video_dir.glob('*.jpg')) + list(video_dir.glob('*.png'))
                frames = sorted(frames, key=lambda x: int(x.stem) if x.stem.isdigit() else 0)
                
                if len(frames) > 0:
                    self.video_data.append({'obj_id': obj_id, 'label': label, 'frames': frames})
        
        print(f"Найдено {len(self.video_data)} видео для обработки.")

    def __len__(self):
        return len(self.video_data)

    def __getitem__(self, idx):
        item = self.video_data[idx]
        frames_paths = item['frames']

        final_frames = []
        
        if len(frames_paths) >= self.num_frames:
            indices = np.linspace(0, len(frames_paths) - 1, self.num_frames, dtype=int)
            selected_paths = [frames_paths[i] for i in indices]
            final_frames = selected_paths
        else:
            final_frames = list(frames_paths)
            padding_needed = self.num_frames - len(final_frames)
            last_frame = final_frames[-1]
            final_frames.extend([last_frame] * padding_needed)

        frames_tensor = []
        for f_path in final_frames:
            img = Image.open(f_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames_tensor.append(img)
            
        frames_tensor = torch.stack(frames_tensor)
        
        return frames_tensor, item['label'] 

def get_test_transforms():
    return T.Compose([
        T.Resize(224),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def create_test_dataloader(
    csv_path='/kaggle/input/datasets/mariaspasyuk/huawei-test/d83a0ce6-dc87-46a6-9679-98a71cf91886.csv',
    frames_root='/kaggle/input/datasets/mariaspasyuk/hwei-eyes',
    num_frames=16,
    batch_size=8,
    num_workers=2
):
    dataset = HuaweiTestDataset(csv_path=csv_path, frames_root=frames_root, num_frames=num_frames, transform=get_test_transforms())
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    return dataset, dataloader

hw_dataset, hw_dataloader = create_test_dataloader()

Найдено 45 видео для обработки.


In [ ]:
model.load_state_dict(torch.load('/kaggle/working/checkpoints_finetuned/model_fine-tuned_epoch_11.pth'))
test_loss, test_acc, test_f1, test_auc = validate_epoch(model, test_loader, criterion, device)
print(f"Test Results: Loss={test_loss:.4f} Acc={test_acc:.4f} F1={test_f1:.4f} AUC={test_auc:.4f}")

In [28]:
model.load_state_dict(torch.load('/kaggle/working/checkpoints_finetuned/model_fine-tuned_epoch_12.pth'))
test_loss, test_acc, test_f1, test_auc = validate_epoch(model, test_loader, criterion, device)
print(f"Test Results: Loss={test_loss:.4f} Acc={test_acc:.4f} F1={test_f1:.4f} AUC={test_auc:.4f}")

Val: 100%|██████████| 58/58 [00:50<00:00,  1.14it/s]

Test Results: Loss=0.4589 Acc=0.8567 F1=0.8708 AUC=0.9271


In [14]:
%cd /kaggle/working/

/kaggle/working


In [15]:
ls

checkpoints/  checkpoints_finetuned/  runs/


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = ResNetLSTM(freeze_cnn=True).to(device)
model.load_state_dict(torch.load('/kaggle/working/checkpoints_finetuned/model_fine-tuned_epoch_11.pth'))
hw_test_loss, hw_test_acc, hw_test_f1, hw_test_auc = validate_epoch(model, hw_dataloader, criterion, device)
print(f"Test Results: Loss={hw_test_loss:.4f} Acc={hw_test_acc:.4f} F1={hw_test_f1:.4f} AUC={hw_test_auc:.4f}")

Using device: cuda


Val:   0%|          | 0/6 [00:00<?, ?it/s]

Test Results: Loss=0.5860 Acc=0.7333 F1=0.7600 AUC=0.8039


In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
model = ResNetLSTM(freeze_cnn=True).to(device)
model.load_state_dict(torch.load('/kaggle/working/checkpoints_finetuned/model_fine-tuned_epoch_12.pth'))
hw_test_loss, hw_test_acc, hw_test_f1, hw_test_auc = validate_epoch(model, hw_dataloader, criterion, device)
print(f"Test Results: Loss={hw_test_loss:.4f} Acc={hw_test_acc:.4f} F1={hw_test_f1:.4f} AUC={hw_test_auc:.4f}")

Using device: cuda


Val:   0%|          | 0/6 [00:00<?, ?it/s]

Test Results: Loss=0.5613 Acc=0.7333 F1=0.7778 AUC=0.8211


In [21]:
import os
import shutil
from IPython.display import FileLink

folder_path = '/kaggle/working/checkpoints_finetuned'
archive_path = '/kaggle/working/checkpoints_finetuned_eyes_archived'

shutil.make_archive(archive_path, 'zip', folder_path)

print(f"Архив создан: {archive_path}.zip")
display(FileLink(f"{archive_path}.zip"))

Архив создан: /kaggle/working/checkpoints_finetuned_eyes_archived.zip


/kaggle/working/checkpoints_finetuned_eyes_archived.zip

# Уменьшим dropout

In [42]:
import os
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

print("Fine-tuning (selective unfreezing)")

model = ResNetLSTM(lstm_hidden=256,lstm_layers=2,dropout=0.3,freeze_cnn=True, unfreeze_layers=1).to(device)

print("layer1 requires_grad:", next(model.layer1.parameters()).requires_grad)
print("layer2 requires_grad:", next(model.layer2.parameters()).requires_grad)
print("layer3 requires_grad:", next(model.layer3.parameters()).requires_grad)
print("layer4 requires_grad:", next(model.layer4.parameters()).requires_grad)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-6, weight_decay=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
CHECKPOINT_DIR = Path("checkpoints_finetuned_dropout03")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
tb_writer = SummaryWriter(log_dir='runs/fine-tuned_cnn_dropout03')

best_val_auc = 0.0
train_loss_1, train_acc_1 = [], []
val_loss_1, val_acc_1, val_f1_1, val_auc_1 = [], [], [], []
EPOCHS = 15

for epoch in range(EPOCHS):
    torch.cuda.empty_cache()
    train_loss, train_acc = train_epoch_ft(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, val_auc = validate_epoch(model, val_loader, criterion, device)
    
    train_loss_1.append(round(train_loss, 4)); train_acc_1.append(round(train_acc, 4))
    val_loss_1.append(round(val_loss, 4)); val_acc_1.append(round(val_acc, 4))
    val_f1_1.append(round(val_f1, 4)); val_auc_1.append(round(val_auc, 4))
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    tb_writer.add_scalar('Stage2/Loss/Train', train_loss, epoch+1)
    tb_writer.add_scalar('Stage2/Accuracy/Train', train_acc, epoch+1)
    tb_writer.add_scalar('Stage2/Learning_Rate', current_lr, epoch+1)
    tb_writer.add_scalar('Stage2/Loss/Val', val_loss, epoch+1)
    tb_writer.add_scalar('Stage2/Accuracy/Val', val_acc, epoch+1)
    tb_writer.add_scalar('Stage2/F1/Val', val_f1, epoch+1)
    tb_writer.add_scalar('Stage2/AUC/Val', val_auc, epoch+1)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Acc={train_acc:.4f}, "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model_fine-tuned.pth")
        
    torch.save(model.state_dict(), CHECKPOINT_DIR / f"model_fine-tuned_epoch_{epoch+1}.pth")

tb_writer.close()
print("Логи сохранены в папку `runs/fine-tuned_cnn`")

Fine-tuning (selective unfreezing)
Разморожено слоев: 1
   (['layer4'])
layer1 requires_grad: False
layer2 requires_grad: False
layer3 requires_grad: False
layer4 requires_grad: True


Val: 100%|██████████| 11/11 [00:20<00:00,  1.86s/it]


Epoch 1: Train Loss=0.6905 Acc=0.5779, Val Loss=0.6857 Acc=0.6340 F1=0.7544 AUC=0.7632


Val: 100%|██████████| 11/11 [00:20<00:00,  1.85s/it]


Epoch 2: Train Loss=0.6824 Acc=0.6493, Val Loss=0.6744 Acc=0.6686 F1=0.7639 AUC=0.7906


...

Epoch 13: Train Loss=0.4302 Acc=0.8761, Val Loss=0.4760 Acc=0.8357 F1=0.8606 AUC=0.9031


Val: 100%|██████████| 11/11 [00:19<00:00,  1.82s/it]


Epoch 14: Train Loss=0.4221 Acc=0.8769, Val Loss=0.4747 Acc=0.8300 F1=0.8571 AUC=0.9038


Val: 100%|██████████| 11/11 [00:20<00:00,  1.83s/it]

Epoch 15: Train Loss=0.4106 Acc=0.8826, Val Loss=0.4827 Acc=0.8271 F1=0.8565 AUC=0.9016
Логи сохранены в папку `runs/fine-tuned_cnn`


In [51]:
model.load_state_dict(torch.load('/kaggle/working/checkpoints_finetuned_dropout03/model_fine-tuned_epoch_15.pth'))
hw_test_loss, hw_test_acc, hw_test_f1, hw_test_auc = validate_epoch(model, hw_dataloader, criterion, device)
print(f"Test Results: Loss={hw_test_loss:.4f} Acc={hw_test_acc:.4f} F1={hw_test_f1:.4f} AUC={hw_test_auc:.4f}")

Val: 100%|██████████| 6/6 [00:03<00:00,  1.75it/s]

Test Results: Loss=0.5892 Acc=0.7556 F1=0.7925 AUC=0.8125
